# acquire

> pull the web, papers, video, files, code and JSON APIs into the vault, once or on a schedule

In [ ]:
#| default_exp acquire

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

Every method is one [fossick](https://github.com/vedicreader/fossick) call plus `Vault.add`. fossick fetches; the vault files with provenance.


In [ ]:
#| export
import hashlib, json, re, time, uuid, warnings
from urllib.parse import urlparse
from fastcore.all import AttrDict, L, Path, chunked, first, patch
from fossick import json_records
from litesearch import code_exts, dir2files, pdf_parse, DOC_EXTS
from vishalakshi.core import Vault, KINDS, is_sanskrit_file
from vishalakshi.jobs import Queue, Retry


In [ ]:
#| export
def clip(s:str, n:int=120) -> str:
    'Collapse whitespace and clip a scraped title to something a breadcrumb can carry.'
    return re.sub(r'\s+', ' ', (s or '').strip())[:n] or 'untitled'

def md_title(md:str, fallback:str='') -> str:
    "First markdown heading in `md`, else `fallback`: scraped <title>s are often junk."
    m = re.search(r'^#{1,2} +(.+)$', md or '', flags=re.M)
    return clip(m.group(1) if m else fallback)

`url(..., auto=True)` retries past bot-wall challenge pages (HTTP 200 with no article). `web` stores the discovery query in meta so `sources()` still explains why a page is here.


In [ ]:
#| export
@patch
def url(self:Vault,
        url:str,            # page to read
        title:str=None,     # defaults to the page's first heading, else its path
        sel:str=None,       # CSS selector to narrow the page before conversion
        kind:str='web',
        auto:bool=True,     # escalate plain -> heavy -> stealthy -> logged-in Chrome past bot walls
        meta:dict=None,
        force:bool=False,
        **kw                # forwarded to fossick.fetch (verify=, headers=, heavy=…)
) -> dict:
    'Fetch one URL, convert it to markdown and file it in the vault.'
    from fossick import fetch, to_md
    pg = fetch(url, sel=sel, auto=auto, **kw)
    md, st = (to_md(pg, sel=sel) if pg is not None else ''), getattr(pg, 'status', None)
    if not md.strip() or (st or 200) >= 400:            # a failed fetch is None, and a bot wall is a 4xx
        return dict(url=url, skipped=f'could not read the page (status {st})', status=st)
    m = dict(meta or {}, url=url, status=st, fetched_at=time.time())
    return dict(self.add(md, title or md_title(md, urlparse(url).path.rsplit('/', 1)[-1] or url),
                         source=url, kind=kind, meta=m, force=force), url=url)

@patch
def crawl(self:Vault, start_url:str, max_pages:int=10, sel:str=None, **kw) -> L:
    'Crawl a docs site or blog from a start URL and file every page in the vault.'
    from fossick import crawl as _crawl, to_md
    pgs = L((pg.url, to_md(pg, sel=sel)) for pg in _crawl(start_url, sel=sel, max_pages=max_pages, **kw))
    return L(self.add(md, md_title(md, u), source=u, kind='web',
                      meta=dict(url=u, via='crawl', root=start_url, fetched_at=time.time()))
             for u, md in pgs if md.strip())

@patch
def web(self:Vault,
        query:str,          # what to search for
        n:int=5,            # top results to read
        google:bool=False,  # real Google ranking via a stealth browser (slower)
        chars:int=60000,    # max markdown chars kept per source
        **kw                # forwarded to fossick.research
) -> AttrDict:
    'Search the web, read the top `n` results, and file all of them in the vault.'
    from fossick import research
    res = research(query, n=n, engine='google' if google else 'search', chars=chars, **kw)
    srcs = L(res['sources']).filter(lambda s: s['md'].strip() and s['href'])
    added = srcs.map(lambda s: dict(self.add(s['md'], clip(s['title']), source=s['href'], kind='web',
                                             meta=dict(url=s['href'], query=query, fetched_at=time.time())),
                                    url=s['href']))
    # dropped pages: bot walls vs empty results; only the former is worth retrying
    return AttrDict(query=query, n_found=len(res['sources']), added=added,
                    dropped=L(res.get('dropped') or []))


In [ ]:
#| eval: false
# Live-scraped search: it needs the network, it is rate-limited, and it returns nothing when
# throttled, which then fails the *next* cell on an empty index
v=Vault(':memory:')
v.web('what is consciousness?', google=True)

/Users/71293/code/personal/orgs/vishalakshi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[2026-08-11 06:57:52] INFO: Fetched (200) <GET https://www.google.com/search?q=what+is+consciousness%3F&hl=en&num=17&sei=TDt6ap_5Fsqc4-EP1fOf2AM> (referer: https://www.google.com/)
[2026-08-11 06:57:53] INFO: Fetched (200) <GET https://en.wikipedia.org/wiki/Consciousness> (referer: https://www.google.com/)
[2026-08-11 06:57:53] INFO: Fetched (200) <GET https://pmc.ncbi.nlm.nih.gov/articles/PMC5924785/> (referer: https://www.google.com/)
[2026-08-11 06:57:53] INFO: Fetched (200) <GET https://mcgovern.mit.edu/2024/04/29/what-is-consciousness/> (referer: https://www.google.com/)
[2026-08-11 06:57:53] INFO: Fetched (200) <GET https://www.reddit.com/r/askphilosophy/comments/kar7as/what_is_consciousness_exactly_a

```python
{ 'added': [{'doc_id': '97aaa07e0c0a3b44', 'title': 'Consciousness', 'kind': 'web', 'nodes': 9, 'chunks': 140, 'url': 'https://en.wikipedia.org/wiki/Consciousness'}, {'doc_id': '0d51d0f4dd5d84f6', 'title': 'What is consciousness exactly and why do so many people ...', 'kind': 'web', 'nodes': 2, 'chunks': 3, 'url': 'https://www.reddit.com/r/askphilosophy/comments/kar7as/what_is_consciousness_exactly_and_why_do_so_many/'}, {'doc_id': '9b5b9514d7b82195', 'title': 'Human Consciousness: Where Is It From and What Is It for', 'kind': 'web', 'nodes': 18, 'chunks': 135, 'url': 'https://pmc.ncbi.nlm.nih.gov/articles/PMC5924785/'}, {'doc_id': '872731547af5c248', 'title': 'What Is Consciousness?', 'kind': 'web', 'nodes': 2, 'chunks': 47, 'url': 'https://www.nature.com/articles/d41586-018-05097-x'}, {'doc_id': '84b9a0d41ce5faa8', 'title': 'What is consciousness? - MIT McGovern Institute', 'kind': 'web', 'nodes': 7, 'chunks': 19, 'url': 'https://mcgovern.mit.edu/2024/04/29/what-is-consciousness/'}],
  'dropped': [],
  'n_found': 5,
  'query': 'what is consciousness?'}
```

In [ ]:
#| eval: false
v.db.t.store.fts_search('consciousness')[0]['content']

"Some philosophers believe that Block's two types of consciousness are not the end of the story. William Lycan, for example, argued in his book _Consciousness and Experience_ that at least eight clearly distinct types of consciousness can be identified (organism consciousness; control consciousness; consciousness _of_ ; state/event consciousness; reportability; introspective consciousness; subjective consciousness; self-consciousness)—and that even this list omits several more obscure forms.[57]\n\n"

### Routing: what a target is

`grab` inspects the target and picks the verb: arXiv id, YouTube URL, GitHub repo, PDF, local file, directory, or bare URL. One call for a CLI or an agent.


In [ ]:
#| export
@patch
def arxiv(self:Vault, id_or_url:str, save_dir:str=None, force:bool=False, **kw) -> dict:
    'Read an arXiv paper (metadata + full text) into the vault as `kind="arxiv"`.'
    if (v := self.route('arxiv')) is not self: return v.arxiv(id_or_url, save_dir=save_dir, force=force, **kw)
    from fossick import read_arxiv
    p = read_arxiv(id_or_url, save_dir=save_dir or str(self.assets('pdfs')), force=force, **kw)
    md = f"# {p['title']}\n\n{p.get('summary','')}\n\n{p.get('source') or ''}"
    m=dict(authors=list(p.get('authors') or []),published=p.get('published'),pdf_path=p.get('pdf_path'),fetched_at=time.time())
    return self.add(md, clip(p['title']), source=p.get('link') or id_or_url, kind='arxiv', force=force,meta=m)

@patch
def pdf(self:Vault, path_or_url:str, title:str=None, force:bool=False, **kw) -> dict:
    'Read a PDF (local path or URL) into the vault, one tree node per heading.'
    from fossick import get_pdf
    if (p:=Path(path_or_url)).exists(): return self.add_file(p, title=title, kind='pdf', force=force)
    if (doc := get_pdf(path_or_url, **kw)) is None: return dict(source=path_or_url, skipped='not a PDF or could not be fetched')
    stem = path_or_url.rsplit('/', 1)[-1].split('?')[0]
    return self.add(list(enumerate(pdf_parse(doc, out_path=self.assets(stem or 'pdf')))),
                    title or clip(stem), source=path_or_url, kind='pdf', force=force,
                    meta=dict(url=path_or_url, fetched_at=time.time()))

@patch
def youtube(self:Vault, url:str, force:bool=False) -> dict:
    "Read a YouTube video's transcript and metadata into the vault."
    from fossick import read_yt
    v = read_yt(url, force=force)
    if not (v.get('source') or '').strip(): return dict(source=url,skipped='no transcript', title=v.get('title'))
    md = f"# {v['title']}\n\n{v.get('description','')}\n\n## Transcript\n\n{v['source']}"
    meta=dict(url=url, channel=v.get('channel'), duration=v.get('duration'), upload_date=v.get('upload_date'), fetched_at=time.time())
    return self.add(md, clip(v['title']), source=url, kind='youtube', force=force,meta=meta)

_GH_BLOB = re.compile(r'https?://github\.com/([^/]+)/([^/]+)/blob/[^/]+/(.+)')

@patch
def github(self:Vault, url:str, **kw) -> AttrDict:
    'File a whole GitHub repo: prose to the vault, source to kosha, through `add_tree`.'
    from fossick import gh_clone
    return self.add_tree(gh_clone(url), **kw)

@patch
def gh_file(self:Vault, url:str, title:str=None, **kw) -> dict:
    'File one file from GitHub: prose into the vault, source into a kosha-indexed directory.'
    from fossick import read_gh_file
    if not (m := _GH_BLOB.match(url)): return dict(source=url, skipped='not a GitHub file URL')
    owner, repo, path = m.groups()
    p = Path(path)
    if p.suffix.lower() in code_exts.split(','):
        from fossick import gh_clone
        d = gh_clone(f'https://github.com/{owner}/{repo}')
        return dict(source=url, kind='code', path=str(d/path), code=self.index_code(d))
    return dict(self.add(read_gh_file(url), title or clip(p.name), source=url, kind='web',
                         meta=dict(url=url, repo=f'{owner}/{repo}', fetched_at=time.time()), **kw), url=url)

def what_is(target:str) -> str:
    """Which kind of thing a `grab` target names: `dir`, `file`, `sanskrit`, `arxiv`, `youtube`, `github`, `ghfile`, `pdf` or `web`."""
    if (p:=Path(target)).is_dir(): return 'dir'
    if p.exists(): return 'sanskrit' if is_sanskrit_file(p) else 'file'
    if 'arxiv.org' in target or re.fullmatch(r'\d{4}\.\d{4,5}(v\d+)?', target): return 'arxiv'
    if re.search(r'youtube\.com|youtu\.be', target): return 'youtube'
    if re.match(r'https?://github\.com/[^/]+/[^/]+', target): return 'ghfile' if '/blob/' in target else 'github'
    path = urlparse(target).path.rstrip('/').lower()
    if path.endswith('.pdf') or path.endswith('/pdf'): return 'pdf'
    if target.startswith('http'): return 'web'
    raise ValueError(f'not a URL, an arXiv id, a file or a directory: {target}')

@patch
def grab(self:Vault,
         target:str,        # a URL, an arXiv id, a YouTube link, a GitHub repo or file, a PDF, a local file or a directory
         title:str=None,
         sel:str=None,      # CSS selector, for the web cases
         shelf:str=None,    # shelf to file it on; None -> whichever `KIND_SHELF` names for its kind
         crawl:bool=False,  # follow links from a web target instead of reading the one page
         max_pages:int=10,  # pages to visit when crawling
         **kw               # forwarded to whichever method the target names
):
    'File anything, by looking at what it is: the one call a CLI or an agent needs.'
    kind = what_is(target)
    v = self.shelf(shelf) if shelf else self.route(kind)
    if crawl:              return v.crawl(target, max_pages=max_pages, sel=sel, **kw)
    if kind == 'dir':      return v.add_tree(target, **kw)
    if kind in ('file', 'sanskrit'): return v.add_file(target, title=title, **kw)
    if kind == 'arxiv':    return v.arxiv(target, **kw)
    if kind == 'youtube':  return v.youtube(target, **kw)
    if kind == 'github':   return v.github(target, **kw)
    if kind == 'ghfile':   return v.gh_file(target, title=title, **kw)
    if kind == 'pdf':      return v.pdf(target, title=title, **kw)
    return v.url(target, title=title, sel=sel, **kw)

@patch
def code(self:Vault, dir:str, types:str=code_exts, **kw) -> L:
    "File a source tree into the vault as `kind='code'`, so code and prose answer one query."
    return self.add_dir(dir, types=types, kind='code', **kw)

In [ ]:
#| hide
# What a target *is* decides which method reads it, so the table is worth testing directly.
test_eq(what_is('https://github.com/AnswerDotAI/fastcore'), 'github')
test_eq(what_is('https://github.com/AnswerDotAI/fastcore/tree/master/nbs'), 'github')
test_eq(what_is('https://github.com/AnswerDotAI/fastcore/blob/master/fastcore/basics.py'), 'ghfile')
test_eq(what_is('https://github.com/AnswerDotAI'), 'web')          # a user page is not a repo
test_eq(what_is('https://arxiv.org/abs/1706.03762'), 'arxiv')
test_eq(what_is('https://example.com/paper.pdf'), 'pdf')
test_eq(what_is('https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=CELEX:32017R0745'), 'pdf')

# The checkout is fossick's to hand over: `gh_clone` returns the working tree, and a local
# path passes through
from fossick import gh_clone
from subprocess import run as _run
from tempfile import mkdtemp
rd = Path(mkdtemp()).resolve()
(rd/'pkg').mkdir()
(rd/'pkg'/'m.py').write_text('def f(): return 1\n')
_run(['git', 'init', '-q', str(rd)], check=True)
test_eq(gh_clone(str(rd)), rd)
(rd/'pkg'/'m.py').unlink()                                         # nothing kosha would index
test_eq(gh_clone(str(rd)), rd)                                     # still the checkout

# `grab` dispatches on the kind, and `crawl=True` overrides it for any web target
calls = []
saved = {n: getattr(Vault, n) for n in ('crawl', 'github', 'gh_file', 'url')}
for n in saved: setattr(Vault, n, (lambda n: lambda self, *a, **kw: calls.append((n, a, kw)))(n))
try:
    g = Vault(':memory:', offline=True)
    g.grab('https://github.com/AnswerDotAI/fastcore');                  test_eq(calls[-1][0], 'github')
    g.grab('https://github.com/AnswerDotAI/fastcore/blob/master/x.py'); test_eq(calls[-1][0], 'gh_file')
    g.grab('https://example.com/post');                                 test_eq(calls[-1][0], 'url')
    g.grab('https://example.com/docs/', crawl=True, max_pages=3)
    test_eq(calls[-1][0], 'crawl')
    test_eq(calls[-1][2]['max_pages'], 3)
finally:
    for n, f in saved.items(): setattr(Vault, n, f)

# `gh_file` splits on the extension: prose is a document, source is something for kosha to index
import fossick
_saved = fossick.read_gh_file
fossick.read_gh_file = lambda url: '# Title\n\nfused ranks share no vector space\n'
try:
    vg = Vault(str(Path(mkdtemp()).resolve()/'v.db'), offline=True)
    r = vg.gh_file('https://github.com/o/r/blob/main/docs/README.md')
    test_eq(r['kind'], 'web')
    assert vg.search('fused ranks')
    test_eq(vg.gh_file('https://github.com/o/r')['skipped'], 'not a GitHub file URL')
    # index the file in fossick's clone so kosha resolves pkg.g
    rr = Path(mkdtemp()).resolve()/'r'
    (rr/'pkg').mkdir(parents=True)
    (rr/'pkg'/'g.py').write_text('def g():\n    return 2\n')
    _run(['git', 'init', '-q', str(rr)], check=True)
    _gc, fossick.gh_clone = fossick.gh_clone, lambda url: rr
    try:
        r = vg.gh_file('https://github.com/o/r/blob/main/pkg/g.py')
        test_eq(r['kind'], 'code')
        test_eq(Path(r['path']), rr/'pkg'/'g.py')          # the file in the checkout, not a copy
        assert Path(r['path']).read_text().startswith('def g')
        assert r['code']                                    # kosha indexed the repo, not one orphan
    finally: fossick.gh_clone = _gc
finally: fossick.read_gh_file = _saved


parse files from /private/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpd4fetc3m/r: 100%|██████████| 1/1 [00:00<00:00, 1250.17it/s]


One directory, two indexes. `add_dir` files documents and `index_code` fills kosha; `add_tree` is both. `grab` on a directory goes here too.


In [ ]:
#| export
@patch
def add_tree(self:Vault,
             dir:str,                # tree to ingest
             types:str=DOC_EXTS,     # extensions filed into the vault as prose
             code:bool=True,         # index source files with kosha, when the tree has any
             kind:str=None,          # override the kind for the prose half
             connect:bool=False,     # rebuild the entity graph at the end; see below
             verbose:bool=False,
             queue:bool=False,       # enqueue the work instead of doing it; `poll` or `drain` runs it
             batch:int=50,           # files per job when queueing
             **kw                    # forwarded to add_file
) -> AttrDict:
    '''Ingest a whole tree, each half to the index that can actually answer questions about it.'''
    p = Path(dir)
    if not p.is_dir(): raise ValueError(f'not a directory: {dir}')
    if queue: return self.enqueue_tree(p, types=types, code=code, kind=kind, batch=batch,
                                       connect=connect, **kw)
    docs = self.add_dir(p, types=types, kind=kind, **kw)
    srcs = dir2files(p, types=code_exts) if code else L()
    out = AttrDict(dir=str(p), docs=docs, n_docs=len(docs), n_code=len(srcs), code=None)
    if srcs:
        try: out.code = self.index_code(p, verbose=verbose)
        except Exception as e:
            warnings.warn(f'could not index {len(srcs)} source files with kosha '
                          f'({type(e).__name__}: {str(e)[:120]}); filing them as prose instead, so '
                          f'they are at least searchable. Install kosha for symbol search.')
            out.code = dict(error=f'{type(e).__name__}: {str(e)[:200]}', filed_as_prose=len(srcs))
            out.docs = docs + srcs.map(self.add_file, kind='code', **kw)
    if connect and (out.n_docs or out.code): out.graph = self.connect()
    return out

### Fanning an ingest out

A tree is one job per `batch` files, not one job for the tree. A crash costs the batch in flight
instead of everything already done, and `jobs(kind='ingest')` says how much is left. The unit is a
batch rather than a file because `add_files` parses in parallel and embeds 2000 chunks at a time;
one job per file would give that up.

Re-running a fan-out is cheap: litesearch skips a source it already holds, so a retried batch does
the work it missed and nothing else. That is what makes at-least-once delivery safe here.

`connect=True` becomes a job of its own rather than something the last batch does. The queue has no
dependency edges, so the graph job sorts behind the batches on priority and raises `Retry` while any
ingest is still pending. That waits on the backoff schedule, and dead-letters after 8 attempts if
the ingest never finishes, which is visible in `dead()` instead of a graph that silently missed a
batch.

In [ ]:
#| export
def _batch_key(paths) -> str:
    'A stable name for a batch of paths, so enqueueing the same batch twice while it is pending is a no-op.'
    return hashlib.sha256('\n'.join(sorted(paths)).encode()).hexdigest()[:16]

@patch
def enqueue_files(self:Vault,
                  files,            # paths to ingest
                  kind:str=None,    # override the kind inferred from the extension
                  batch:int=50,     # files per job; one job is one `add_files` call
                  route:bool=True,  # send Sanskrit sources to the Sanskrit shelf
                  priority:int=0,
                  **kw              # forwarded to add_files
) -> L:
    'Enqueue an ingest as one job per `batch` files, so a crash resumes at the last batch.'
    fs = L(files).map(Path)
    if not fs: return L()
    if route and self.name == 'store':
        sa, rest = L(), L()
        for f in fs: (sa if is_sanskrit_file(f) else rest).append(f)
        groups = [('sanskrit', sa), (self.name, rest)]
    else: groups = [(self.name, fs)]
    out = L()
    for store, g in groups:
        for ch in chunked(g, batch):
            ps = [str(p) for p in ch]
            out.append(self.q.enqueue('ingest', dict(store=store, kind=kind, files=ps, kw=kw),
                                      priority=priority, key=f'ingest:{store}:{_batch_key(ps)}'))
    return out

@patch
def enqueue_tree(self:Vault, dir, types:str=DOC_EXTS, code:bool=True, kind:str=None,
                 batch:int=50, connect:bool=False, **kw) -> AttrDict:
    'Enqueue a whole tree: one job per batch of documents, one more for the code half.'
    p, out = Path(dir), L()
    docs = dir2files(p, types=types)
    out += self.enqueue_files(docs, kind=kind, batch=batch, **kw)
    srcs = dir2files(p, types=code_exts) if code else L()
    if srcs: out.append(self.q.enqueue('index_code', dict(dir=str(p)), key=f'index_code:{p}'))
    # priority 1 sorts it behind the batches, and the handler waits for the ones it did not outrank.
    # the key has no tree in it because the graph is of the whole vault: one pending job covers every
    if connect and out: out.append(self.q.enqueue('connect', {}, priority=1, key='connect',
                                                  max_attempts=8))
    return AttrDict(dir=str(p), n_docs=len(docs), n_code=len(srcs), queued=len(out),
                    jobs=list(out.attrgot('id')))

@patch
def _job_ingest(self:Vault, payload:dict) -> dict:
    'Queue handler for one batch of files.'
    v = self if payload['store'] == self.name else self.shelf(payload['store'])
    fs = L(payload['files']).map(Path).filter(lambda p: p.exists())
    if not fs: return dict(skipped='every file in the batch is gone')
    docs = v.add_files(fs, kind=payload.get('kind'), **(payload.get('kw') or {}))
    return dict(store=payload['store'], files=len(fs), added=len(docs))

@patch
def _job_index_code(self:Vault, payload:dict) -> dict:
    "Queue handler for the kosha half of a tree. A kosha failure dead-letters rather than filing source as prose."
    return dict(dir=payload['dir'], code=self.index_code(payload['dir']))

@patch
def _job_connect(self:Vault, payload:dict) -> dict:
    'Queue handler rebuilding the entity graph, once the ingests it should cover have landed.'
    # the graph is of the whole vault, so any pending ingest is one this should wait for
    if (n := self.q.pending('ingest')): raise Retry(f'{n} ingest jobs still to run')
    return dict(graph=self.connect())

In [ ]:
#| hide
# a queued tree is one job per batch, and draining it lands the same documents as the direct path
from tempfile import mkdtemp
d = Path(mkdtemp())
for i in range(5): (d/f'n{i}.md').write_text(f'# note {i}\n\nranks fuse across shelf {i}\n')
vq = Vault(':memory:', offline=True)
r  = vq.add_tree(d, queue=True, batch=2, code=False)
test_eq(r['n_docs'], 5)
test_eq(r['queued'], 3)                       # 5 files, 2 per job
test_eq(len(vq.jobs(kind='ingest')), 3)
test_eq(vq.stats()['docs'], 0)                # nothing has run yet
ran = vq.q.drain('w')
test_eq(len(ran), 3)
test_eq({x['status'] for x in ran}, {'ok'})
test_eq(vq.stats()['docs'], 5)
assert vq.search('ranks fuse')

# enqueueing the same tree again while it is pending does not double the work
vq2 = Vault(':memory:', offline=True)
test_eq(vq2.add_tree(d, queue=True, batch=2, code=False)['queued'], 3)
test_eq(vq2.add_tree(d, queue=True, batch=2, code=False)['queued'], 3)
test_eq(len(vq2.jobs(kind='ingest')), 3)      # the second call found the first call's jobs


In [ ]:
#| hide
# the point of the fan-out: a worker killed partway through resumes at the batch it was holding
vq3 = Vault(':memory:', offline=True)
vq3.add_tree(d, queue=True, batch=2, code=False)
t = time.time()
first(vq3.q.claim('doomed', now=t))            # claimed, then the worker dies without acking
vq3.q.drain('w2')                              # the survivor takes what it can reach
test_eq(vq3.stats()['docs'], 3)                # two files still locked under the dead lease
test_eq(vq3.q.stats()['running'], 1)
test_eq(len(vq3.q.reclaim(now=t + vq3.q.lease + 1)), 1)
vq3.q.drain('w2', now=time.time() + 3600)      # the reclaimed batch backs off before it retries
test_eq(vq3.stats()['docs'], 5)                # ...and the lost batch is picked up, not lost
test_eq(vq3.q.stats(), dict(ready=0, running=0, done=3, dead=0, next_due=None))


In [ ]:
#| hide
# connect=True is a job, and it waits for the batches rather than rebuilding a graph that misses them
vq4 = Vault(':memory:', offline=True)
r = vq4.add_tree(d, queue=True, batch=2, code=False, connect=True)
test_eq(r['queued'], 4)                                  # 3 batches and the graph
test_eq(vq4.q.pending('ingest'), 3)
test_eq(first(vq4.q.claim('w', n=9)).kind, 'ingest')     # the graph job sorts behind them

# reached while a batch is still pending, it retries instead of running early
vq5 = Vault(':memory:', offline=True)
vq5.add_tree(d, queue=True, batch=2, code=False, connect=True)
vq5.q.claim('holder', n=1)                               # one batch claimed and never finished
cj = first(vq5.q.claim('w', n=9).filter(lambda j: j['kind'] == 'connect'))
test_eq(vq5.q.run_one(cj)['status'], 'retry')
test_eq(first(vq5.q.jobs(kind='connect'))['state'], 'ready')

# with nothing left pending it runs, and the graph is there
vq6 = Vault(':memory:', offline=True)
vq6.add_tree(d, queue=True, batch=2, code=False, connect=True)
ran = vq6.q.drain('w', now=time.time()+86400)
test_eq([x['status'] for x in ran].count('ok'), 4)
test_eq(vq6.q.pending(), 0)
assert 'graph' in first(x for x in ran if x['kind'] == 'connect')['result']


### Harvest: read a page's API, not its HTML

fossick captures the calls a page makes; the vault picks the one carrying records, follows pagination, and files each record as its own section. `apis` lists candidates; `harvest` / `add_records` file them.


In [ ]:
#| export
def records_md(recs, title_keys=('name', 'title', 'displayName', 'productName', 'label', 'sku', 'id')) -> str:
    'Records as markdown, one `##` section per record, so each becomes its own retrievable node.'
    def ttl(r): return clip(next((r[k] for k in title_keys if isinstance(r.get(k), str) and r[k].strip()),
                                 json.dumps(r, default=str)), 80)
    return '\n\n'.join(f'## {ttl(r)}\n\n```json\n{json.dumps(r, indent=1, default=str)}\n```'
                       for r in L(recs).map(lambda r: r if isinstance(r, dict) else dict(value=r)))

In [ ]:
#| export
@patch
def apis(self:Vault,
         url:str,             # page to watch
         pattern:str='*',     # glob/regex filtering captured request URLs
         session:bool=False,  # capture through the logged-in debug Chrome
         preview:int=240,     # chars of each response shown
         **kw                 # forwarded to fossick.find_xhr
) -> L:
    'Discover the JSON endpoints a page calls, so you can read its data instead of its HTML.'
    from fossick import find_xhr
    self._caps = L(find_xhr(url, pattern=pattern, session=session, **kw))
    return L(AttrDict(n=i, url=h['url'], content_type=h.get('content_type'),
                      records=len(json_records(h.get('data'))),
                      preview=json.dumps(h.get('data'), default=str)[:preview])
             for i, h in enumerate(self._caps))

@patch
def harvest(self:Vault,
            url:str,                # the page whose API you want
            pattern:str='*',        # which captured request URLs to keep
            title:str=None,         # document title; defaults to the page host + path
            capture:int=None,       # replay a specific endpoint from the last apis() call
            pages:int=1,            # pages to pull; >1 paginates the endpoint
            page_field:str='page',  # query/body key incremented per page
            session:bool=False,     # capture through the logged-in Chrome
            force:bool=False,
            **kw
) -> dict:
    "Sniff a page's JSON API, pull the records, and file them in the vault as `kind='data'`."
    from fossick import replay_xhr, paginate_api
    caps = getattr(self, '_caps', None)
    if capture is None or not caps:
        found = self.apis(url, pattern=pattern, session=session)
        if not found: return dict(url=url, skipped='no JSON endpoints captured')
        capture, caps = max(found, key=lambda h: h.records).n, self._caps
    cap, hit = caps[capture].get('capture'), caps[capture]
    ep = (cap or hit)['url']
    try:one, key = json_records(replay_xhr(cap, **kw).json() if cap else hit.get('data'), with_key=True)
    except Exception as e: return dict(url=url, endpoint=ep, skipped=f'{type(e).__name__}: {e}')
    if pages > 1 and one: kw.setdefault('page_size', len(one))
    items = (paginate_api(ep, payload=cap.get('request_body') if cap else None, page_field=page_field,
      results_field=key, method=(cap.get('method') or 'GET').upper() if cap else 'GET', max_pages=pages, **kw)
             if pages > 1 else one)
    if not items: return dict(url=url, endpoint=ep, skipped='no records found in the response')
    ttl = title or f'{urlparse(url).netloc}{urlparse(url).path}'.strip('/')
    return dict(self.add_records(items, ttl, source=ep, force=force, meta=dict(page=url, endpoint=ep,
           harvested_at=time.time())), endpoint=ep, records=len(items))

@patch
def add_records(self:Vault, recs:list, title:str, source:str=None, kind:str='data', force:bool=False,
                meta:dict=None) -> dict:
    'File a list of dicts you already have (any API, any export) as one document, one section each.'
    return self.add(f'# {title}\n\n{records_md(recs)}', title, source=source or f'records:{title}',
                    kind=kind, force=force, meta=dict(meta or {}, records=len(L(recs))))

In [ ]:
#| hide
# The rows here are nested under `data.products`, which is where `apis` counted them, while
# pagination looked only at the top-level keys and found nothing, so `pages=1` filed 3 records
import fossick
from fossick import json_records
_pay = {'meta': {'total': 3}, 'data': {'products': [{'sku': 'A'}, {'sku': 'B'}, {'sku': 'C'}]}}
_cap = dict(url='https://shop.example/api/products', method='GET', request_body=None)

_h = Vault(':memory:', offline=True)
_h._caps = [dict(url=_cap['url'], data=_pay, capture=_cap)]
_seen = {}
_sv = fossick.replay_xhr, fossick.paginate_api
fossick.replay_xhr = lambda cap, **kw: AttrDict(json=lambda: _pay)
fossick.paginate_api = lambda url, **kw: (_seen.update(kw), _pay['data']['products'])[1]
try:
    r1 = _h.harvest(_cap['url'], capture=0)
    test_eq(r1['records'], 3)
    r2 = _h.harvest(_cap['url'], capture=0, pages=2, force=True)
    test_eq(r2['records'], 3)                       # the same rows, not a skipped run
    test_eq(_seen['results_field'], 'products')     # pagination is told which list to read
finally: fossick.replay_xhr, fossick.paginate_api = _sv

test_eq(_h.doc(_cap['url'])['meta']['records'], 3)  # and each row is its own section in the vault
assert _h.search('sku')

# A session-less sniff records the *response* but not the request, so `find_xhr` returns no
# `capture` at all, and refusing to file the rows it already holds made every default-path
_n = Vault(':memory:', offline=True)
_n._caps = [dict(url=_cap['url'], data=_pay)]        # exactly what `session=False` leaves behind
_r = _n.harvest('https://shop.example/listing', capture=0, title='sniffed')
test_eq((_r['records'], _r['endpoint']), (3, _cap['url']))
assert _n.search('sku')

# ...and page 2 is asked for at the size the first response actually was
_seen.clear()
_sv2 = fossick.paginate_api
fossick.paginate_api = lambda url, **kw: (_seen.update(kw), _pay['data']['products'])[1]
try: _n.harvest('https://shop.example/listing', capture=0, title='sniffed', pages=3, force=True)
finally: fossick.paginate_api = _sv2
test_eq((_seen['page_size'], _seen['max_pages'], _seen['method']), (3, 3, 'GET'))

### Watches: keeping it current

An action is an acquisition method name. `watch` schedules it, `poll` runs what is due. A due watch
is enqueued rather than run on the spot, so a failure retries with backoff instead of waiting a
whole interval, and a worker killed mid-fetch loses its lease and not the job. See [jobs](11_jobs.ipynb).

`next_run` advances from the time a run was *scheduled*, so a slow fetch does not push the schedule
later every cycle. After a gap the missed intervals are counted and `catchup` of them are run, the
rest recorded in `missed`: coming back from a week offline should not start a week of crawls at once.
`remind` writes a note instead of fetching anything.

In [ ]:
#| export
ACTIONS = ('url', 'web', 'harvest', 'arxiv', 'youtube', 'crawl', 'path', 'remind')

_DUR, _MULT = re.compile(r'([\d.]+)\s*([smhdw])', re.I), dict(s=1, m=60, h=3600, d=86400, w=604800)

def secs(every) -> float:
    "Seconds from `'30m'`, `'6h'`, `'2 days'`, `'1w'`, `'1h30m'`, or a number of seconds."
    try: return float(every)
    except (TypeError, ValueError): pass
    if not (ms := _DUR.findall(str(every))): raise ValueError(f'not a duration: {every!r}')
    return sum(float(n) * _MULT[u.lower()] for n, u in ms)

@patch(as_prop=True)
def q(self:Vault) -> Queue:
    "The vault's job queue, on its database, with the acquisition handlers registered."
    if (q := getattr(self.db, '_vq', None)) is None: q = self.db._vq = Queue(self.db)
    # a shelf may reach it first; the main store rebinds the handlers to itself when it arrives
    if self.name == 'store' or not q.handlers:
        for k, fn in (('watch', self._job_watch), ('ingest', self._job_ingest),
                      ('index_code', self._job_index_code), ('connect', self._job_connect)):
            q.register(k, fn)
    return q

@patch
def _w(self:Vault):
    'The watches table, created on first use.'
    t = self.db.t.watches
    t.create(id=str, action=str, target=str, params=str, every=float, note=str, enabled=int,
             last_run=float, last_status=str, next_run=float, runs=int, catchup=int, missed=int,
             pk='id', if_not_exists=True)
    have = set(t.columns_dict)
    for c, d in dict(catchup='INTEGER DEFAULT 1', missed='INTEGER DEFAULT 0').items():
        if c not in have: self.db.conn.execute(f'ALTER TABLE watches ADD COLUMN {c} {d}')
    return t

@patch
def watch(self:Vault,
          target:str,         # URL, query, arXiv id, or the text of a reminder
          action:str='url',   # one of ACTIONS: what to do when it fires
          every:str='1d',     # interval: '30m', '6h', '1d', '1w', or seconds
          note:str=None,      # why you are watching
          start:float=None,   # first run time (epoch); defaults to now
          catchup:int=1,      # missed intervals to make up after a gap; the rest count as `missed`
          **params            # forwarded to the action (n=, pattern=, pages=, sel=, ...)
) -> dict:
    'Register a recurring job: re-read a page, re-run a search, re-harvest an API, or remind you.'
    assert action in ACTIONS, f'action must be one of {ACTIONS}'
    row = dict(id=uuid.uuid4().hex[:12], action=action, target=target, params=json.dumps(params),
               every=secs(every), note=note or '', enabled=1, next_run=start or time.time(), runs=0,
               catchup=max(1, catchup), missed=0)
    self._w().insert(row, replace=True)
    return dict(row, params=params)

@patch
def watches(self:Vault, due_only:bool=False, at:float=None) -> L:
    'Every registered watch, soonest first; `due_only` keeps the ones whose next run has arrived.'
    where = f'enabled=1 AND next_run<={time.time() if at is None else at}' if due_only else None
    return L(self._w()(where=where, order_by='next_run')).map(
        lambda r: dict(r, params=json.loads(r['params'] or '{}')))

@patch
def _watch_row(self:Vault, wid:str) -> dict|None:
    'One watch by id, with its params parsed.'
    r = first(self._w()(where='id=?', where_args=[wid]))
    return dict(r, params=json.loads(r['params'] or '{}')) if r else None

@patch
def unwatch(self:Vault, watch_id:str):
    'Delete a watch. The documents it already filed stay in the vault.'
    self._w().delete(watch_id)

@patch
def pause(self:Vault, watch_id:str, enabled:bool=False):
    'Disable (or re-enable) a watch without losing it.'
    self._w().update(dict(id=watch_id, enabled=int(enabled)))

@patch
def _do_watch(self:Vault, w:dict):
    'Perform one watch action. Raises on failure, and `Retry` where the failure is worth another try.'
    params = dict(w['params'])
    if w['action'] in ('url', 'arxiv', 'youtube', 'path'): params.setdefault('force', True)
    res = (self.note(w['target'], title=w.get('note') or None, tags=['reminder'])
           if w['action'] == 'remind' else self.grab(w['target'], **params)
           if w['action'] == 'path' else getattr(self, w['action'])(w['target'], **params))
    # a bot wall is a skip worth retrying; 'no transcript' is a skip that will never change
    if isinstance(res, dict) and res.get('skipped') and (res.get('status') or 0) >= 400:
        raise Retry(res['skipped'])
    return res

@patch
def _job_watch(self:Vault, payload:dict) -> dict:
    'Queue handler for a due watch: run it, and record the outcome on its row. `next_run` is the scheduler\'s.'
    if (w := self._watch_row(payload['watch_id'])) is None:
        return dict(skipped=f"watch {payload['watch_id']} is gone")
    def _done(st): self._w().update(dict(id=w['id'], last_run=time.time(), last_status=st, runs=w['runs']+1))
    try: res = self._do_watch(w)
    except Exception:
        _done('error'); raise
    _done('skipped' if isinstance(res, dict) and res.get('skipped') else 'ok')
    return res

def _intervals(w:dict, now:float) -> int:
    'Whole intervals of `w` that have come and gone by `now`. 1 when it has only just come due.'
    return int((now - w['next_run']) // max(w['every'], 1.)) + 1

@patch
def _advance(self:Vault,
             w:dict,        # the watch row as it was read
             n:int,         # intervals that came due, from `_intervals`
             ran:int        # how many of them are being run; the rest are counted as missed
):
    'Move `next_run` past the intervals that came due, from when they were due rather than from now.'
    self._w().update(dict(id=w['id'], next_run=w['next_run'] + n * max(w['every'], 1.),
                          missed=(w['missed'] or 0) + n - ran))

@patch
def run_watch(self:Vault, w:dict) -> dict:
    'Fire one watch now, off the queue, and record the outcome. Failures come back, they do not raise.'
    t0 = time.time()
    try:
        res = self._job_watch(dict(watch_id=w['id']))
        status = 'skipped' if isinstance(res, dict) and res.get('skipped') else 'ok'
    except Exception as e: res, status = dict(error=f'{type(e).__name__}: {str(e)[:200]}'), 'error'
    # a due watch run by hand consumes its slot, or it stays due and the next poll runs it again
    if w['next_run'] <= t0: self._advance(w, _intervals(w, t0), 1)
    return dict(watch_id=w['id'], action=w['action'], target=w['target'], status=status,
                took=round(time.time()-t0, 2), result=res)

@patch
def schedule(self:Vault, at:float=None) -> L:
    'Enqueue a job for every watch that has come due, and advance its next run without drift.'
    now, out = (time.time() if at is None else at), L()
    for w in self.watches(due_only=True, at=now):
        n = _intervals(w, now)
        take = max(1, min(n, w['catchup'] or 1))
        for i in range(take):
            sched = w['next_run'] + (n - take + i) * max(w['every'], 1.)   # the most recent `take`
            out.append(self.q.enqueue('watch', dict(watch_id=w['id'], scheduled_at=sched),
                                      key=f"{w['id']}@{int(sched)}"))
        self._advance(w, n, take)
    return out

@patch
def poll(self:Vault, at:float=None, limit:int=None, connect:bool=True, worker:str='poll') -> dict:
    'Reclaim, schedule and drain. This is the tick a scheduler, a cron or a frontend calls.'
    back = self.q.reclaim()
    self.schedule(at=at)
    ran = self.q.drain(worker, limit=limit or 1000)
    if connect and ran.filter(lambda r: r['status'] == 'ok'): self.connect()
    pending = self.watches()
    return dict(checked=len(pending), ran=len(ran), results=list(ran), reclaimed=len(back),
                dead=self.q.stats()['dead'], next_due=pending[0]['next_run'] if pending else None)

@patch
def jobs(self:Vault, state:str=None, kind:str=None, limit:int=50) -> L:
    'Jobs in the vault queue; `state="dead"` is the ones that used up their attempts.'
    return self.q.jobs(state=state, kind=kind, limit=limit)

@patch
def retry_job(self:Vault, job_id:str) -> dict:
    'Put a dead or pending job back on the queue now, with its attempt count reset.'
    return self.q.retry(job_id)

In [ ]:
#| hide
import time as _t

# `remind` is the one action with no network in it, so the whole path runs here
v9 = Vault(':memory:', offline=True)
w9 = v9.watch('check the kiln', action='remind', every='1h')
test_eq(len(v9.watches(due_only=True)), 1)
r = v9.poll(connect=False)
test_eq(r['ran'], 1)
test_eq(r['results'][0]['status'], 'ok')
test_eq(v9.watches()[0]['runs'], 1)
test_eq(v9.watches()[0]['last_status'], 'ok')
test_eq(v9.q.stats()['done'], 1)
# the schedule advances from when the run was *due*, so a slow run does not push the next one later
test_close(v9.watches()[0]['next_run'], w9['next_run'] + 3600, eps=1e-3)

# the direct path still works, and it consumes the slot it ran rather than leaving the watch due
v9b = Vault(':memory:', offline=True)
w9b = v9b.watch('check the kiln', action='remind', every='1h')
test_eq(v9b.run_watch(v9b.watches()[0])['status'], 'ok')
test_eq(len(v9b.watches(due_only=True)), 0)
test_close(v9b.watches()[0]['next_run'], w9b['next_run'] + 3600, eps=1.)
test_eq(v9b.q.pending(), 0)                                # nothing was queued; it ran on the spot
# ...but a manual run before it is due does not disturb the schedule
before = v9b.watches()[0]['next_run']
test_eq(v9b.run_watch(v9b.watches()[0])['status'], 'ok')
test_eq(v9b.watches()[0]['next_run'], before)


In [ ]:
#| hide
# ten hours offline on an hourly watch: one run, and the nine that were skipped are counted
v10 = Vault(':memory:', offline=True)
t0  = _t.time() - 10*3600
v10.watch('kiln', action='remind', every='1h', start=t0)
test_eq(len(v10.schedule()), 1)
row = v10.watches()[0]
test_eq(row['missed'], 10)
test_close(row['next_run'], t0 + 11*3600, eps=1.)
assert row['next_run'] > _t.time(), 'the next run is in the future, however long the gap was'

# catchup=3 makes up the three most recent intervals and counts the rest
v11 = Vault(':memory:', offline=True)
t1  = _t.time() - 10*3600
v11.watch('kiln', action='remind', every='1h', start=t1, catchup=3)
sched = v11.schedule()
test_eq(len(sched), 3)
test_eq(v11.watches()[0]['missed'], 8)
test_eq([round(j['payload']['scheduled_at'] - t1) for j in sched], [8*3600, 9*3600, 10*3600])


In [ ]:
#| hide
# a bot wall retries with backoff instead of waiting out the interval; a dead end does not
v12 = Vault(':memory:', offline=True)
v12.url = lambda t, **kw: dict(skipped='could not read the page (status 403)', status=403)
v12.watch('https://x.test/a', action='url', every='1d')
v12.schedule()
test_eq(first(v12.q.drain('w'))['status'], 'retry')
j = v12.q.jobs(state='ready')[0]
test_eq(j['attempt'], 1)
assert j['run_at'] > _t.time(), 'the retry is minutes away, not a day'
assert j['run_at'] < _t.time() + 86400

v12.url = lambda t, **kw: dict(skipped='not a PDF')
v12.db.q('UPDATE jobs SET run_at=0')
test_eq(first(v12.q.drain('w'))['status'], 'skipped')
test_eq(v12.q.stats()['ready'], 0)

# a watch deleted while its job sits in the queue is a skip, not a failure
v13 = Vault(':memory:', offline=True)
w13 = v13.watch('kiln', action='remind', every='1h')
v13.schedule()
v13.unwatch(w13['id'])
test_eq(first(v13.q.drain('w'))['status'], 'skipped')


## Try it

In [ ]:
v = Vault(':memory:')
v.add_records([dict(sku='A1', name='Free range eggs', price=4.5),
               dict(sku='B2', name='Oat milk', price=2.1)], 'dairy')
v.search('eggs')[0]['breadcrumb']

'dairy › Free range eggs'

In [ ]:
# fossick's json_records walk; harvest pagination uses the same
_pay = {'data': {'items': [{'a': 1}, {'a': 2}, {'a': 3}]}}
test_eq(len(json_records(_pay)), 3)
test_eq(json_records(_pay, with_key=True)[1], 'items')
test_eq(secs('6h'), 21600); test_eq(secs('1w'), 604800); test_eq(secs(90), 90)
test_eq(secs('30m'), 1800); test_eq(secs('2 days'), 172800); test_eq(secs('45s'), 45)
test_eq(secs('1h30m'), 5400); test_eq(secs('90'), 90)
test_fail(lambda: secs('whenever'), contains='not a duration')
w = v.watch('late chunking', action='web', every='1d', n=3)
test_eq(w['params'], dict(n=3))
test_eq(len(v.watches(due_only=True)), 1)
v.unwatch(w['id'])
test_eq(len(v.watches()), 0)


In [ ]:
#| hide
# a mixed tree: a README and a source file, the shape of any repo
from tempfile import mkdtemp
d = Path(mkdtemp())
(d/'README.md').write_text('# fuse\n\nRanks are fused because the legs share no vector space.')
(d/'fuse.py').write_text('def fuse(ranks):\n    "Reciprocal rank fusion over ranked lists."\n    return ranks\n')

r = Vault(':memory:').add_tree(d, connect=False)
test_eq((r.n_docs, r.n_code), (1, 1))                 # the doc to the vault, the source to kosha
test_eq(r.docs.attrgot('title'), ['README'])
assert r.code and not r.code.get('error'), r.code     # kosha answered, not the prose fallback

# code=False leaves the source where it is, and a tree with no source never reaches kosha at all
r = Vault(':memory:').add_tree(d, code=False, connect=False)
test_eq((r.n_code, r.code), (0, None))
test_fail(lambda: Vault(':memory:').add_tree(d/'README.md'), contains='not a directory')

# the graph is rebuilt once at the end, not once per file
r = Vault(':memory:').add_tree(d, connect=True)
assert r.graph['entities'] > 0, r.graph

parse files from /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpeekzjq36:   0%|          | 0/1 [00:00<?, ?it/s]

parse files from /var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpeekzjq36: 100%|██████████| 1/1 [00:00<00:00, 671.63it/s]


In [ ]:
#| hide
# `arxiv` routes itself, because the *method* fixes the kind and `grab` would route it anyway
v2 = Vault(':memory:')
test_eq(v2.route('arxiv').name, 'papers')
# ...and nothing else does
v2.add_records([dict(sku='A1', name='free range eggs')], 'groceries')
test_eq(v2.doc('records:groceries')['title'], 'groceries')       # right here, where `find` looks
assert v2.search('free range eggs')
(d/'b.py').write_text('def fuse(x):\n    return x\n')
v2.code(d)
assert 'b' in v2.sources(kind='code').attrgot('title')   # filed here, as `kind='code'` prose
# an explicit shelf is never overruled: what was asked for is what happens
test_eq(v2.shelf('sanskrit', offline=True).route('arxiv').name, 'sanskrit')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()